# Data Science 1 · Linear and logistic regression

**Lesson:** Data Science 1, Day 1 · **Time:** about 15 minutes · **ISLP:** 3.1–3.3, 4.1–4.3 · **Data:** CADets release 1

**What you will do**
- Predict length of stay: first with the mean, then with a line by age decade
- Split the variation in length of stay into what the line explains and the error, and get R²
- Go from the odds ratio of a 2×2 table to a logistic regression, and calculate a patient's risk of death

Run each cell in order with **Shift + Enter**. Cells marked **Your turn** have a blank (`____`) to fill in.

In [ ]:
# Setup: load the CADets course dataset. Run this cell first.
# RELEASE: 1 from Day 1, 2 from Day 2, 3 from Day 5. Use the release participants have on your lesson day.
RELEASE = 1

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FILES = {1: "cadets_admissions_release1_core.csv",
         2: "cadets_admissions_release2_intervention.csv",
         3: "cadets_admissions_release3_full.csv"}
DATA_URL = "https://raw.githubusercontent.com/Rotem-Gershon/cadets-notebooks/main/data/"

def load_cadets(release=RELEASE):
    name = FILES[release]
    if os.path.exists(name):  # a copy uploaded to this session
        return pd.read_csv(name)
    try:  # the course repository, from the release day on
        return pd.read_csv(DATA_URL + name)
    except Exception:
        pass
    try:  # before the release day: upload the file from the course website's dataset page
        from google.colab import files
        print(f"Release {release} is not published yet. Upload {name} from the course website's dataset page.")
        files.upload()
        return pd.read_csv(name)
    except ImportError:
        raise FileNotFoundError(f"Put {name} in the same folder as this notebook.")

df = load_cadets()
print(f"Release {RELEASE}: {df.shape[0]:,} admissions, {df.shape[1]} columns")

## Step 1 · From the mean to a line

Without a model, the best guess for every patient is the mean length of stay. Then we look at age, in decades (8 = age 80 to 89).

In [ ]:
import statsmodels.formula.api as smf

df["age_decade"] = df["age"] // 10
print("Mean length of stay:", round(df["los_days"].mean(), 2), "days")
df.groupby("age_decade")["los_days"].agg(["count", "mean"]).round(2)

The mean stay rises with age. `smf.ols` fits the straight line closest to all the patients (least squares).

In [ ]:
model_age = smf.ols("los_days ~ age_decade", data=df).fit()
print(model_age.params.round(3))

plt.scatter(df["age_decade"], df["los_days"], s=6, alpha=0.2)
plt.axhline(df["los_days"].mean(), ls="--", color="grey", label="Mean")
decades = pd.DataFrame({"age_decade": [0, 10]})
plt.plot(decades["age_decade"], model_age.predict(decades), color="red", label="Regression line")
plt.xlabel("Age decade"); plt.ylabel("Length of stay (days)"); plt.legend(); plt.show()

- **Intercept (1.08):** the predicted stay at age 0, where the line meets the vertical axis.
- **Slope (0.70):** each additional decade of age adds 0.7 days.

### Your turn 1

Use the equation to predict the stay of a patient in their sixties: replace the blank with the age decade.

In [ ]:
patient = pd.DataFrame({"age_decade": [____]})
print("Predicted stay:", round(model_age.predict(patient)[0], 2), "days")

## Step 2 · How much does the line explain? R²

Each stay = the mean + what the line adds + the error. Square the parts and add them over all patients.

In [ ]:
mean_los = df["los_days"].mean()
predicted = model_age.fittedvalues

total = ((df["los_days"] - mean_los) ** 2).sum()
explained = ((predicted - mean_los) ** 2).sum()
error = ((df["los_days"] - predicted) ** 2).sum()

print(f"Total {total:,.0f} = explained {explained:,.0f} + error {error:,.0f}")
print("R² = explained / total =", round(explained / total, 3))
print("R² from the model      =", round(model_age.rsquared, 3))

Age decade explains 8% of the variation in length of stay.

## Step 3 · Many predictors: binary, categorical and continuous

Heart failure is binary (0/1), admission diagnosis is categorical (`C(...)` makes one 0/1 column per group, compared with cardiac), systolic blood pressure is continuous.

In [ ]:
model_los = smf.ols("los_days ~ age_decade + heart_failure + C(admit_diagnosis_group) + systolic_bp",
                    data=df).fit()
print(model_los.params.round(3))
print("R²:", round(model_los.rsquared, 3))

In [ ]:
patient = pd.DataFrame({"age_decade": [8], "heart_failure": [1],
                        "admit_diagnosis_group": ["infection"], "systolic_bp": [110]})
print("Predicted stay:", round(model_los.predict(patient)[0], 2), "days")

## Step 4 · A yes/no outcome: from the odds ratio to logistic regression

The 2×2 table of heart failure and in-hospital death, as in the epidemiology refresher:

In [ ]:
table = pd.crosstab(df["heart_failure"], df["in_hospital_death"])
odds_hf = table.loc[1, 1] / table.loc[1, 0]
odds_no_hf = table.loc[0, 1] / table.loc[0, 0]
odds_ratio = odds_hf / odds_no_hf
print(table)
print("OR:", round(odds_ratio, 2), "   log(OR):", round(np.log(odds_ratio), 3))

A logistic regression with heart failure as its only predictor. Its coefficient is log(OR), and its intercept is the log-odds without heart failure.

In [ ]:
model_hf = smf.logit("in_hospital_death ~ heart_failure", data=df).fit(disp=0)
print(model_hf.params.round(3))
print("exp(β) =", round(np.exp(model_hf.params["heart_failure"]), 2))

Risk: add up z from the coefficients, then pass it through the logistic function, risk = 1 / (1 + e^(−z)).

In [ ]:
b0, b1 = model_hf.params["Intercept"], model_hf.params["heart_failure"]
for hf in [0, 1]:
    z = b0 + b1 * hf
    print(f"Heart failure = {hf}: z = {z:.2f}, risk = {1 / (1 + np.exp(-z)):.3f}")
print(df.groupby("heart_failure")["in_hospital_death"].mean().round(3))

The model's risks are exactly the death rates in the two rows of the table.

## Step 5 · Age, then many predictors

A continuous predictor: the coefficient is per decade of age.

In [ ]:
model_dec = smf.logit("in_hospital_death ~ age_decade", data=df).fit(disp=0)
print(model_dec.params.round(3))
print("OR per decade:", round(np.exp(model_dec.params["age_decade"]), 2))
print(model_dec.predict(pd.DataFrame({"age_decade": [5, 8]})).round(3))

Several predictors: z is a linear combination of all of them. `np.exp` of the coefficients gives the adjusted ORs.

In [ ]:
model_death = smf.logit("in_hospital_death ~ age_decade + heart_failure + resp_rate + spo2",
                        data=df).fit(disp=0)
print(pd.DataFrame({"β": model_death.params, "OR": np.exp(model_death.params)}).round(3))

### Your turn 2

A patient in their seventies, without heart failure, with respiratory rate 22. Fill in an SpO₂ value of 94 and run.

In [ ]:
patient = pd.DataFrame({"age_decade": [7], "heart_failure": [0], "resp_rate": [22], "spo2": [____]})
print("Risk of in-hospital death:", round(model_death.predict(patient)[0], 3))

The same model gives every admission a risk. The ten highest:

In [ ]:
df["predicted_risk"] = model_death.predict(df)
columns = ["patient_id", "age", "heart_failure", "resp_rate", "spo2", "predicted_risk", "in_hospital_death"]
df.sort_values("predicted_risk", ascending=False).head(10)[columns].round(2)

## Key points

- Without a model the best guess is the mean. A regression line does better: intercept = the prediction at 0, slope = the change per unit.
- R² is the share of the variation around the mean that the model explains.
- More predictors, of any type (binary, categorical, continuous), extend the same equation.
- For a yes/no outcome, logistic regression models the log-odds. With one predictor, the coefficient is log(OR).
- Risk = 1 / (1 + e^(−z)), where z is the sum of the coefficients times the patient's values.

## Further reading

- James, Witten, Hastie, Tibshirani and Taylor. *An Introduction to Statistical Learning with Applications in Python* (ISLP), sections 3.1–3.3 and 4.1–4.3. Free at [statlearning.com](https://www.statlearning.com).
- Enrichment notebook: scaling and significance in linear regression; the logit, odds ratios and the road to AUROC.